In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import datetime, timezone
import pathlib

import exifread
import holoviews as hv
import hvplot.pandas
import numpy as np
import pandas as pd
import rawpy

In [ ]:
data_dir = pathlib.Path("~/allsky/raw").expanduser()

In [ ]:
@dataclass
class ExposureInfo:
    exposure_time: float
    original_datetime: datetime
    median: float

In [ ]:
def get_exposure_info(data_file: pathlib.Path):
    with data_file.open("rb") as ifile:
        tags = exifread.process_file(ifile)

    exp_time = eval(str(tags["EXIF ExposureTime"]))
    original_datetime = datetime.strptime(str(tags["EXIF DateTimeOriginal"]), "%Y:%m:%d %H:%M:%S").replace(tzinfo=timezone.utc)

    with rawpy.imread(str(data_file)) as raw:
        rgb = raw.postprocess(no_auto_bright=True, use_auto_wb=False, gamma=(1, 1), output_bps=16)
        luminance = np.dot(rgb[..., :3], [0.299, 0.587, 0.114])
        median = np.median(luminance)

    return ExposureInfo(exp_time, original_datetime, median)

In [ ]:
data_files = sorted(list(data_dir.glob("*.cr2")))

In [ ]:
with ThreadPoolExecutor() as executor:
    results = list(executor.map(get_exposure_info, data_files))

In [ ]:
df = pd.DataFrame(results)
# df = df.loc[(df["original_datetime"] <= '2025-12-22')]

In [ ]:
df.hvplot.scatter("original_datetime", "median", by=["exposure_time"], hover_cols=["exposure_time"], legend=False)